In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import math
import time

c:\Users\offsh\Documents\project\dl\pt_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [3]:
class FastTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes=2, d_model=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model, num_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, padding_mask):
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        
        src_key_padding_mask = (padding_mask == 0)
        
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        
        mask_expanded = padding_mask.unsqueeze(-1).expand(x.size()).float()
        sum_embeddings = torch.sum(x * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        return self.classifier(pooled_output)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
dataset = load_dataset("stanfordnlp/imdb")

def collate_fn(batch):
    texts = [item['text'] for item in batch]
    labels = [item['label'] for item in batch]
    
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=512, return_tensors='pt')
    
    return encodings['input_ids'], encodings['attention_mask'], torch.tensor(labels)

c:\Users\offsh\Documents\project\dl\pt_env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\offsh\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\offsh\Documents\project\dl\pt_env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarn

In [7]:
train_loader = DataLoader(dataset['train'], batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(dataset['test'], batch_size=32, shuffle=False, collate_fn=collate_fn)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FastTransformerClassifier(vocab_size=tokenizer.vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [9]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0
    start_time = time.time()
    
    for i, (input_ids, mask, labels) in enumerate(train_loader):
        input_ids, mask, labels = input_ids.to(device), mask.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, mask)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predictions = outputs.argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        if (i + 1) % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {i+1}/{len(train_loader)} | Loss: {loss.item():.4f} | Acc: {correct/total:.2f}")
            
    print(f"--- Epoch {epoch+1} Complete | Time: {time.time() - start_time:.2f}s | Train Acc: {correct/total:.4f} ---")

model.eval()
correct, total = 0, 0
print("Running Evaluation on Test Set...")
with torch.no_grad():
    for input_ids, mask, labels in test_loader:
        input_ids, mask, labels = input_ids.to(device), mask.to(device), labels.to(device)
        outputs = model(input_ids, mask)
        predictions = outputs.argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"Final Test Accuracy: {correct/total:.4f}")

Epoch 1 | Batch 100/782 | Loss: 0.5423 | Acc: 0.58
Epoch 1 | Batch 200/782 | Loss: 0.4763 | Acc: 0.66
Epoch 1 | Batch 300/782 | Loss: 0.5110 | Acc: 0.69
Epoch 1 | Batch 400/782 | Loss: 0.4452 | Acc: 0.70
Epoch 1 | Batch 500/782 | Loss: 0.1616 | Acc: 0.72
Epoch 1 | Batch 600/782 | Loss: 0.4617 | Acc: 0.73
Epoch 1 | Batch 700/782 | Loss: 0.4692 | Acc: 0.74
--- Epoch 1 Complete | Time: 160.90s | Train Acc: 0.7488 ---
Epoch 2 | Batch 100/782 | Loss: 0.4852 | Acc: 0.84
Epoch 2 | Batch 200/782 | Loss: 0.2302 | Acc: 0.85
Epoch 2 | Batch 300/782 | Loss: 0.3925 | Acc: 0.85
Epoch 2 | Batch 400/782 | Loss: 0.2610 | Acc: 0.85
Epoch 2 | Batch 500/782 | Loss: 0.4070 | Acc: 0.85
Epoch 2 | Batch 600/782 | Loss: 0.3549 | Acc: 0.85
Epoch 2 | Batch 700/782 | Loss: 0.3509 | Acc: 0.84
--- Epoch 2 Complete | Time: 177.73s | Train Acc: 0.8439 ---
Epoch 3 | Batch 100/782 | Loss: 0.6320 | Acc: 0.88
Epoch 3 | Batch 200/782 | Loss: 0.2107 | Acc: 0.88
Epoch 3 | Batch 300/782 | Loss: 0.3742 | Acc: 0.88
Epoch 3 | B

c:\Users\offsh\Documents\project\dl\pt_env\Lib\site-packages\torch\nn\modules\transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


Final Test Accuracy: 0.8378


In [10]:
def predict_sentiment(text, model, tokenizer, device, max_length=512):
    model.eval() 
    
    encodings = tokenizer(
        text, 
        truncation=True, 
        padding='max_length', 
        max_length=max_length, 
        return_tensors='pt'
    )
    
    input_ids = encodings['input_ids'].to(device)
    mask = encodings['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, mask)
        
        probabilities = F.softmax(outputs, dim=-1)[0]
        prediction = torch.argmax(probabilities).item()
        
    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probabilities[prediction].item() * 100
    
    return sentiment, confidence

In [11]:
custom_review = "I was blown away by the cinematography, but the plot was a bit slow in the middle. Overall, a great experience."

sentiment, confidence = predict_sentiment(custom_review, model, tokenizer, device)
print(f"\nReview: '{custom_review}'")
print(f"Prediction: {sentiment} ({confidence:.1f}% confident)")


Review: 'I was blown away by the cinematography, but the plot was a bit slow in the middle. Overall, a great experience.'
Prediction: Positive (99.7% confident)


In [12]:
torch.save(model.state_dict(), r"models/custom_transformer_imdb.pt")
print("Model saved to custom_transformer_imdb.pt")

Model saved to custom_transformer_imdb.pt
